# Utilizzo dell'SDK di openAI 

from zero to hero!  

i bullets rappresentano argomenti da articolare

In [ ]:
import os
from openai import OpenAI
from IPython.display import Image, display
from pprint import pprint

chiave_api = "inserisci_qua"

**Spiegazioni (provvisoriamente AI Generated)**

In questa sezione vediamo come utilizzare l’SDK di OpenAI per la **generazione di testo**, mettendo in pratica alcuni concetti fondamentali:

---

### 1) Differenze tra *prompt* e *chat completion* 
- **Prompt (vecchia interfaccia “completions”)**: si invia una singola stringa come prompt e si riceve il testo completato dal modello. È più difficile mantenere una conversazione multi-turno (cioè con molte domande-risposte consecutive).
- **Chat completion**: si invia un elenco di messaggi con ruoli diversi (`system`, `user`, `assistant`), ottimizzato per gestire conversazioni interattive e contesti più lunghi.

---

### 2) Differenza tra modelli “instruct” e modelli “chat”
- **Modelli *instruct*** (es. `gpt-3.5-turbo-instruct`): sono stati addestrati per seguire istruzioni date in un prompt singolo. Richiamano la modalità “legacy” dei completions.
- **Modelli *chat*** (es. `gpt-3.5-turbo`, `gpt-4o`): sono addestrati per comprendere e generare risposte in stile conversazionale, basate su più messaggi con ruoli distinti. Generalmente più potenti e versatili per chatbot e dialoghi multi-turno.

---

### 3) Struttura dei *responses* (oggetti e JSON)
Quando chiamiamo il metodo `client.completions.create()` o `client.chat.completions.create()`, la risposta del modello (un oggetto Python o JSON) avrà in comune:
- `choices`: lista di possibili completamenti generati (spesso è di lunghezza 1 se non diversamente specificato).
- `choices[0].text` oppure `choices[0].message.content`: contiene il testo effettivo generato.
- Metadati vari come `usage` (token utilizzati), `model`, `finish_reason` e altri campi.

Nel caso **chat**, la parte da stampare si trova in `choices[0].message.content`.
 **Struttura dell’oggetto di risposta**:  
  ```json
  {
    "id": "chatcmpl-123",
    "object": "chat.completion",
    "created": 1677652288,
    "model": "gpt-4o-mini",
    "system_fingerprint": "fp_44709d6fcb",
    "choices": [
      {
        "index": 0,
        "message": {
          "role": "assistant",
          "content": "Hello there, how may I assist you today?"
        },
        "logprobs": null,
        "finish_reason": "stop"
      }
    ],
    "usage": {
      "prompt_tokens": 9,
      "completion_tokens": 12,
      "total_tokens": 21,
      "completion_tokens_details": {
        "reasoning_tokens": 0,
        "accepted_prediction_tokens": 0,
        "rejected_prediction_tokens": 0
      }
    }
  }
  ```
  - **id** e **object**: identificativo e tipo della risorsa.  
  - **created**: timestamp in secondi.  
  - **model**: il modello effettivamente usato.  
  - **system_fingerprint**: indica internamente la configurazione del servizio.  
  - **choices**: array con uno o più completamenti generati. Ogni completamento ha:
    - **index**: l’indice della scelta  
    - **message**: l’effettivo testo di risposta nel campo `content`, con ruolo solitamente `assistant`.  
    - **finish_reason**: indica se il modello ha raggiunto uno stop token, o se ha terminato per lunghezza massima, ecc.  
  - **usage**: dettagli di token impiegati (`prompt_tokens`, `completion_tokens` e `total_tokens`), utile per analizzare costi e conteggi. `completion_tokens_details` fornisce sottocategorie di come i token sono stati utilizzati in modelli con ragionamento interno (o1, ecc.).


---




In [51]:
# istanzio il client passando la key (cella sopra) come variabile ambiente
os.environ["OPENAI_API_KEY"] = chiave_api
client=OpenAI()

### Prima prova con prompt (modello legacy) e limite delle conversazioni (provvisoriamente AI Generated)
In questa cella eseguiamo:
```python
response = client.completions.create(
    prompt="Raccontami una storia divertente in italiano",
    ...
)
```
stiamo sfruttando un **modello instruct**. Questo approccio funziona ma:
- È meno adatto a conversazioni: ogni richiesta è “scollegata” dalle precedenti.
- Difficile mantenere contesto in più turni di dialogo.

In [ ]:
response = client.completions.create(prompt="Raccontami una storia divertente in italiano",
                                    model = "gpt-3.5-turbo-instruct",
                                    max_tokens=1000,
                                    temperature=1
                                    )

print(response.choices[0].text)



C'era una volta un contadino di nome Giovanni che aveva una mucca che si chiamava Margherita. Ogni giorno, Giovanni si svegliava alle 5 del mattino e andava a mungere la mucca per ottenere il latte fresco per la sua famiglia.

Un giorno, mentre Giovanni stava mungendo Margherita, lei iniziò a muoversi in modo strano e infine, con un grande botto, esplose il latte dappertutto, colpendo Giovanni in pieno nel viso.

Sorpreso e tutto grondante di latte, Giovanni iniziò a ridere a crepapelle. Margherita, pensando che fosse un gioco divertente, iniziò a rimuginare sul prato a saltare e a schizzare latte in ogni direzione.

I vicini, vedendo Giovanni e Margherita in questa situazione folle, si unirono alla festa e iniziarono a lanciarsi scherzosamente latte l'uno contro l'altro.

Dopo un po', tutto il villaggio era coinvolto in una gigantesca battaglia di latte. C'era latte ovunque, sui vestiti, sui volti e perfino sui tetti delle case.

Giovanni e Margherita si divertivano come mai prima d

bozza by me

I modelli di chat, ovvero quelli più potenti e moderni, seguono una sintassi di generazione del testo che invia una "chat" al sistema, il quale deve generare il completamento più logico a quella chat.

Viene inviata al modello una lista di messaggi, ciascun messaggio è un dizionario con chiavi "role" e "content".

I ruoli possono essere "system", "user" o "assistant" o "function". Al momento di occupiamo solo dei primi due:

- I messaggi "system" contengono le istruzioni che vanno "suggerite" al modello prima che generi la risposta. Solitamente si utilizza per le istruzioni o per fornire dati aggiuntivi

- I messaggi "user" contengono la vera e propria domanda che il modello dovrà inferire, e solitamente sono l'ultimo messaggio.

- I messaggi "assistant" contengono una risposta già data dal modello in caso una chat sia già inziata (se siamo oltre il primo scambio domanda-risposta)




### Uso dei modelli di *chat* (provvisoriamente AI Generated)
Come illustrato in questa cella, con `client.chat.completions.create()` inviamo una lista di messaggi:
```python
messages = [
    {"role": "system", "content": messaggio_system},
    {"role": "user", "content": messaggio_user}
]
```
- Il **messaggio di ruolo `system`** (“Sei un algoritmo cantastorie...”) imposta il contesto iniziale: stile e comportamento.
- Il **messaggio di ruolo `user`** rappresenta la richiesta effettiva: “raccontami una storia divertente”.

Il risultato viene recuperato con `response.choices[0].message.content`.

---


In [14]:
messaggio_system = "Sei un algoritmo cantastorie. Parla in napoletano"
messaggio_user = " raccontami una storia divertente"

response = client.chat.completions.create(
    messages = [
        {"role" : "system", "content" : messaggio_system},
        {"role": "user", "content": messaggio_user}
    ],
    model = "gpt-3.5-turbo",
    temperature = 1
)

print(response.choices[0].message.content)


Certo, sienteme...
C'era una vota 'nu guaglione napulitano chiammato Peppinillo ca aveva 'na passione p' 'e babbà. Tutti i giorni, appena se svegliava, correva a 'na pasticceria vicino a casa sua pe mma n'annà nu babbà. Era 'na vera ossessione pe chistu dulce tradizionale napulitano!

Peppinillo s'è reso conto ca stava spesanno troppe quattrini pe st'abitudine, e ha deciso ca doveva truvà 'na suluzione. Accussì, ha deciso d'allerne 'nda cucina e provà a fà i babbà da sulu. Ha raclutato 'nu guardiano 'e viecchj vecchia vicino a 'casa sua pe dargli 'na manu.

Dopo varie prove e 'mperime stento, Peppinillo è riuscito a preparà nu babbà 'e delizioso tutto da sulu! Si sentiva tanto fiero 'e se stesso e contento ca ora poteva esse autossuficiente 'na cosa ca amava 'na maniera assurda.

E così, Peppinillo è diventato famoso 'int' 'o quartiere pe' suoi babbà fatti 'e mano, e ogni matina, invece 'e corre 'na pasticceria, correva alla sua cucina a preparà stu dulce tradizionale cu tutta 'a passi

### Possiamo anche implementare più messaggi system, ad esempio uno con una istruzione e uno con dei dati aggiuntivi


### Esempio di più messaggi *system* (provvisoriamente AI Generated)
Nella parte con “Ludovico Ariosto”, abbiamo due messaggi `system` per fornire ulteriori dati e istruzioni, oltre al messaggio `user` che chiede un riassunto in 50 parole.  
Questo dimostra come **possiamo inserire nel contesto** parti di testo da cui il modello attingerà per costruire la risposta, in modo analogo a un sistema di retrieval.

---

**Riferimenti alla documentazione:**
- Nella sezione *“Text generation”* di OpenAI si descrive come creare prompt e come scegliere tra modelli “instruct” e “chat”.  
- Si ribadiscono i concetti di token, contesto e la differenza tra prompt singolo e messaggi multipli per chat.

In sintesi, il notebook mostra la transizione dal **vecchio completions** (prompt singolo) al **nuovo chat completions** (messaggi multipli e conversazioni), evidenziando vantaggi di quest’ultimo in termini di robustezza e gestione del contesto.

In [15]:
messaggio_system = "Sei un algoritmo che aiuta gli studenti. Fai ciò che ti viene chiesto"


messaggio_system2 = """
Ludovico Ariosto, uno degli scrittori più influenti del Rinascimento italiano, è noto soprattutto per il suo poema epico "Orlando Furioso". Nato il 8 settembre 1474 a Reggio Emilia, Ariosto crebbe in una famiglia benestante grazie alla posizione di suo padre come comandante della fortezza di Reggio. La famiglia si trasferì poi a Ferrara, dove Ludovico trascorse gran parte della sua vita.

La formazione di Ariosto fu dapprima legale, come desiderato dal padre, ma ben presto si orientò verso gli studi umanistici. Studiò sotto il reggente della Scuola d'Este, Gregorio da Spoleto, che gli insegnò greco e latino e gli trasmise l'amore per la letteratura classica. Durante gli anni universitari, Ariosto iniziò a scrivere poesie, influenzato dai lavori di poeti come Virgilio e Ovidio.

Dopo l'università, Ariosto entrò al servizio della corte degli Este a Ferrara, dove rimase per la maggior parte della sua vita lavorativa. Qui, iniziò la sua carriera come diplomatico e poi come capitano della fortezza di Canossa. Durante questo periodo, si dedicò anche alla scrittura e al teatro, producendo commedie che rispecchiavano lo stile e l'umorismo della commedia classica latina e delle opere di Plauto.

La sua opera più celebre, "Orlando Furioso", fu pubblicata per la prima volta nel 1516. Il poema è un ampliamento del lavoro iniziato da Matteo Maria Boiardo con "Orlando Innamorato". "Orlando Furioso" mescola elementi romantici, avventurosi e fantastici, raccontando le storie di numerosi cavalieri, dame, maghi e mostri, con un intreccio che si snoda attraverso vari continenti e sfide eroiche. La narrativa complessa e l'uso di una lingua ricca e variegata fecero di questo poema un capolavoro del Rinascimento e un modello per la letteratura epica successiva.

Ariosto rivedette "Orlando Furioso" due volte, pubblicando edizioni rinnovate nel 1521 e nel 1532, quest'ultima solo un anno prima della sua morte avvenuta il 6 luglio 1533 a Ferrara. Oltre a essere un epico poeta, Ariosto fu anche un abile amministratore e funzionario, incarichi che gli furono spesso gravosi ma che svolse con dedizione.

La vita di Ludovico Ariosto fu segnata dall'equilibrio tra le sue responsabilità alla corte degli Este e il suo impegno letterario. Nonostante le pressioni e le sfide della vita di corte, riuscì a creare opere che hanno lasciato un segno indelebile nella letteratura italiana e mondiale. La sua abilità nel tessere trame complesse, il suo uso magistrale della lingua e la sua profonda comprensione della natura umana lo rendono una figura di spicco del suo tempo.
"""


messaggio_user = "Fammi un riassunto in 50 parole"    # oppure "in quali anni rivise la sua opera?"


response = client.chat.completions.create(
    messages = [

        {"role" : "system", "content" : messaggio_system},
        {"role" : "system", "content" : messaggio_system2},
        {"role": "user", "content": messaggio_user}
    ],
    model = "gpt-4o",
    temperature = 1
)

print(response.choices[0].message.content)

Ludovico Ariosto, nato nel 1474 a Reggio Emilia, è un influente scrittore rinascimentale italiano noto per "Orlando Furioso", un poema epico pubblicato nel 1516. Lavorò alla corte degli Este a Ferrara, equilibrando incarichi diplomatici e letteratura. Morì nel 1533 a Ferrara, lasciando un'impronta duratura nella letteratura.


## Generazione immagini

- menzionare costi
- copyright ?
- boh

In [19]:
response = client.images.generate(
    model="dall-e-3",
    prompt="un piccolo robot di nome LIA con il suo nome scritto sul petto",
    size="1024x1024",
    quality="standard",
    n=1,
)

url_immagine=response.data[0].url
display(Image(url=url_immagine))

## Output strutturati

- esigenza da cui nasce: integrazione in applicazioni
- json schema vs basemodel pydantic
- utilità

**Spiegazione (provvisoriamente AI Generated)**

In questo esempio vediamo come il **Structured Outputs** del modello OpenAI venga impiegato per estrarre e restituire informazioni strutturate su un film, a partire dal testo HTML di una pagina web (in questo caso, la pagina Wikipedia di *Her (2013 film)*).

1. **Definizione di un modello Pydantic**  
   La classe `ParsedMovie` eredita da `BaseModel` di Pydantic e definisce lo schema dei campi che vogliamo estrarre:  
   - `title: str`  
   - `actors: list[str]`  
   - `genre: str`  
   - `won_oscar: bool`  
   - `educational: bool`  
   - `fun_fact: str|None` (opzionale, poiché può essere `None`)  

   Questi campi corrispondono alle informazioni da reperire nel testo della pagina: titolo, attori, genere, se ha vinto o meno l’Oscar, se è adatto in ambito scolastico e un eventuale fatto interessante.

2. **Istruzioni per l’estrazione strutturata**  
   La variabile `instruction` contiene il prompt da dare al modello in ruolo `system`. Qui si chiede esplicitamente di estrarre queste informazioni in italiano. In particolare, le istruzioni indicano quali campi vogliamo (ad esempio “titolo”, “attori”, “genere” etc.) e in che lingua (italiano).

3. **Download della pagina HTML**  
   Viene fatto un `requests.get(url)` alla pagina Wikipedia. Se la risposta ha `status_code` pari a 200 (tutto OK), si procede alla chiamata del modello OpenAI.

4. **Chiamata al modello con `response_format`**  
   La riga:
   ```python
   completion = client.beta.chat.completions.parse(
       model="gpt-4o",
       messages=[
           {"role": "system", "content": instruction},
           {"role": "user", "content": response.text},
       ],
       response_format=ParsedMovie,
   )
   ```
   usa un meccanismo di **structured outputs**: grazie a `response_format=ParsedMovie`, diciamo al modello che la risposta deve essere sempre un JSON aderente allo schema `ParsedMovie`.  
   - `messages` include il messaggio `system` (le istruzioni su come “parlare” e che cosa estrarre) e il messaggio `user`, che contiene l’intero HTML della pagina come stringa.

   Con questa modalità di **parsed** (ovvero `completions.parse`), il modello genera solo un oggetto strutturato conforme a `ParsedMovie`. Se qualche campo fosse mancante o il modello non riuscisse a popolare correttamente lo schema, potremmo rilevare un errore di validazione.

5. **Lettura del risultato**  
   - `event = completion.choices[0].message.parsed` memorizza l’oggetto estratto (di tipo `ParsedMovie`).  
   - `parsed_form = event.model_dump()` converte l’oggetto Pydantic in un dizionario standard Python.  
   - `pprint(parsed_form)` stampa i campi estratti per una lettura più chiara.

**In sintesi**, questo esempio mostra come utilizzare il meccanismo “Structured Outputs” per **fissare uno schema di output** (in questo caso con Pydantic) e avere la certezza che il modello generi risposte in un formato JSON coerente con i campi di interesse. In questo modo possiamo estrarre informazioni specifiche (titolo, attori, genere, ecc.) da testi lunghi o non strutturati – come una pagina HTML.

In [28]:
from pydantic import BaseModel
import requests

class ParsedMovie(BaseModel):
    
    title: str
    actors: list[str]
    genre: str
    won_oscar: bool
    educational: bool
    fun_fact: str|None
    
    
instruction = """Estrai le informazioni su un film dal codice html una pagina web. Dimmi in italiano:
-il titolo
-quali attori compaiono
-quale è il genere principale
-se ha vinto un oscar
-se è adatto per la visione in una scuola
-se presente, un fatto interessante sul film"""

url="https://en.wikipedia.org/wiki/Her_(2013_film)"

response = requests.get(url)
if response.status_code == 200:
    completion = client.beta.chat.completions.parse(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": instruction },
            {"role": "user", "content": response.text},
        ],
        response_format=ParsedMovie,
    )

event = completion.choices[0].message.parsed
parsed_form=event.model_dump()
pprint(parsed_form)

{'actors': ['Joaquin Phoenix',
            'Scarlett Johansson',
            'Amy Adams',
            'Rooney Mara',
            'Olivia Wilde'],
 'educational': False,
 'fun_fact': 'The film was dedicated to four people who had died before its '
             'release: James Gandolfini, Harris Savides, Maurice Sendak, and '
             'Adam Yauch.',
 'genre': 'Romantic Drama',
 'title': 'Her',
 'won_oscar': True}


## Function Calling

- capire se si puo fare in java o via chiamate rest (dubito, trovare workaround)
- difficile ma figo
- descrizione dettagliata sintassi dei tools
- spiegazione responses e ruolo functions
- spiegazione di come si articola il flusso

**Spiegazione (provvisoriamente AI Generated)**

In questa cella mostriamo come sfruttare il **function calling** di un modello OpenAI per gestire prenotazioni di voli, treni e hotel. Il codice è strutturato così:

1. **Simulazione delle funzioni di prenotazione**  
   Abbiamo tre funzioni:  
   - `book_flight()`: simula la prenotazione di un volo.  
   - `book_train()`: simula la prenotazione di un treno.  
   - `book_hotel()`: simula la prenotazione di un hotel.  

   Ciascuna funzione crea un `payload` con i parametri ricevuti (es. destinazione, date, ecc.) e chiama `simulate_request()`, che simula una chiamata HTTP verso un endpoint fittizio. Il risultato viene poi impacchettato e restituito come dizionario Python.

2. **Definizione dei tools**  
   La lista `tools` racchiude le specifiche delle funzioni che vogliamo rendere disponibili al modello, seguendo il **formato JSON schema** richiesto dal function calling.  
   Ogni entry di `tools` contiene:  
   - Un campo `"type": "function"`.  
   - Un oggetto `"function"` che descrive la singola funzione, con:
     - `"name"`: nome della funzione (ad esempio `"book_flight"`).  
     - `"description"`: una descrizione testuale di ciò che fa.  
     - `"parameters"`: un JSON schema che descrive i parametri di ingresso necessari, con i rispettivi tipi, descrizioni e eventuali campi obbligatori.  
   - `"strict": True`, che fa sì che il modello cerchi di rispettare rigidamente lo schema di quei parametri (ad esempio, non aggiunge campi extra e non ne omette di obbligatori).

   Questo approccio si basa su quanto visto nella documentazione: quando il modello riceve un prompt, potrà decidere di **invocare** una di queste funzioni, compilando i parametri nello schema JSON definito, invece di restituire solo testo libero.

3. **La funzione `bot_viaggi()`**  
   Questa funzione simula un vero flusso di conversazione con il modello:
   - Viene creata una lista di messaggi (`messages`) che include un ruolo `system` e uno `user`, secondo lo schema tipico delle chat completions.  
   - Si invoca `client.chat.completions.create(...)` specificando:
     - `model="gpt-4o"`, cioè il modello da utilizzare.  
     - `messages=messages`, ovvero i messaggi di contesto per la conversazione (incluso ciò che chiede l’utente).  
     - `tools=tools`, la lista di funzioni disponibili.  
     - `tool_choice="auto"`, cioè lasciamo al modello la libertà di decidere se chiamare funzioni e quante.
   - Il risultato del modello (`response`) conterrà l’eventuale **chiamata** (`tool_calls`) che il modello desidera fare. Se `tool_calls` non è vuoto, significa che il modello vuole invocare una o più delle funzioni che abbiamo definito.  
   - Se ci sono funzioni da chiamare (passo detto _“function calling”_):  
     1. Aggiungiamo ai messaggi la “richiesta di function call” del modello.  
     2. Eseguiamo le funzioni Python effettive (`book_flight`, `book_train`, `book_hotel`) in base al nome della funzione da chiamare.  
     3. Appendiamo il risultato di ciascuna chiamata come messaggio di ruolo `"tool"`, così il modello può “vedere” l’output e continuare la conversazione.  
   - Si effettua poi un nuovo giro di completamento (`client.chat.completions.create`) con i messaggi aggiornati, permettendo al modello di produrre la risposta finale (inclusi eventuali ulteriori passaggi).  

In questo modo, la logica della prenotazione (ossia l’effettiva esecuzione di un booking) rimane **fuori** dal modello: è Python a compiere le prenotazioni vere (o simularle, in questo caso). Il modello, invece, ha il compito di capire quando chiamare una funzione, qual è il payload corretto da passare e come usare la risposta. Questo è il fulcro dell’**integrazione tra i modelli OpenAI e il vostro backend**: il modello decide **se** e **come** chiamare un tool (funzione), e il vostro codice esegue le azioni necessarie.

In [ ]:
import json


################################################################################
# mock funzioni di prenotazione
################################################################################

def simulate_request(endpoint, payload) -> dict:
    """
    Simula una richiesta HTTP a un servizio esterno, restituendo sempre
    una risposta di successo con gli stessi dati inviati.
    """
    print(f"Simulazione: effettuo una POST a {endpoint} con payload:")
    print(payload)
    return {"status": "success", "details": payload}

def book_flight(destination: str, date: str, passengers: int, flight_class: str = "Economy") -> dict:
    """
    Simula la prenotazione di un volo.
    Restituisce un dizionario Python (che poi convertiremo in stringa).
    """
    endpoint = "https://api.simulatedbooking.com/flights"
    payload = {
        "destination": destination,
        "date": date,
        "passengers": passengers,
        "class": flight_class
    }
    response = simulate_request(endpoint, payload)
    return {
        "message": "Volo prenotato con successo",
        "response": response
    }

def book_train(from_station: str, to_station: str, date: str, time: str) -> dict:
    """
    Simula la prenotazione di un treno.
    """
    endpoint = "https://api.simulatedbooking.com/trains"
    payload = {
        "from": from_station,
        "to": to_station,
        "date": date,
        "time": time
    }
    response = simulate_request(endpoint, payload)
    return {
        "message": "Treno prenotato con successo",
        "response": response
    }

def book_hotel(city: str, check_in_date: str, check_out_date: str, rooms: int) -> dict:
    """
    Simula la prenotazione di un hotel.
    """
    endpoint = "https://api.simulatedbooking.com/hotels"
    payload = {
        "city": city,
        "check_in": check_in_date,
        "check_out": check_out_date,
        "rooms": rooms
    }
    response = simulate_request(endpoint, payload)
    return {
        "message": "Hotel prenotato con successo",
        "response": response
    }

################################################################################
# Definizione dei tool con schema JSON
################################################################################

tools = [
    {
        "type": "function",
        "function": {
            "name": "book_flight",
            "description": "Prenota un volo per una destinazione specifica in una data definita.",
            "parameters": {
  "type": "object",
  "properties": {
    "destination": {
      "type": "string",
      "description": "La destinazione del volo"
    },
    "date": {
      "type": "string",
      "description": "La data del volo (YYYY-MM-DD)"
    },
    "passengers": {
      "type": "number",
      "description": "Il numero di passeggeri"
    },
    "flight_class": {
      "type": ["string", "null"],
      "description": "La classe del volo (es. Economy, Business). Può essere null."
    }
  },
  "required": ["destination", "date", "passengers", "flight_class"],
  "additionalProperties": False
},

            "strict": True
        }
    },
    {
        "type": "function",
        "function": {
            "name": "book_train",
            "description": "Prenota un treno per un percorso specifico in una data e ora prestabilite.",
            "parameters": {
                "type": "object",
                "properties": {
                    "from_station": {
                        "type": "string",
                        "description": "La stazione di partenza"
                    },
                    "to_station": {
                        "type": "string",
                        "description": "La stazione di arrivo"
                    },
                    "date": {
                        "type": "string",
                        "description": "La data del viaggio (YYYY-MM-DD)"
                    },
                    "time": {
                        "type": "string",
                        "description": "L'orario di partenza (HH:MM)"
                    }
                },
                "required": ["from_station", "to_station", "date", "time"],
                "additionalProperties": False
            },
            "strict": True
        }
    },
    {
        "type": "function",
        "function": {
            "name": "book_hotel",
            "description": "Prenota un hotel in una città per un determinato periodo.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "La città in cui prenotare l'hotel"
                    },
                    "check_in_date": {
                        "type": "string",
                        "description": "La data di check-in (YYYY-MM-DD)"
                    },
                    "check_out_date": {
                        "type": "string",
                        "description": "La data di check-out (YYYY-MM-DD)"
                    },
                    "rooms": {
                        "type": "number",
                        "description": "Il numero di stanze richieste"
                    }
                },
                "required": ["city", "check_in_date", "check_out_date", "rooms"],
                "additionalProperties": False
            },
            "strict": True
        }
    }
]

def bot_viaggi(user_message: str) -> None: # impacchetto tutto come funzione per renderlo riutilizzabile nel notebook
    messages = [
        {
            "role": "system",
            "content": (
                "Sei un assistente clienti per prenotazioni viaggi. Puoi prenotare voli, "
                "treni o hotel in base alle richieste degli utenti."
            )
        },
        {
            "role": "user",
            "content": (
                user_message
            )
        }
    ]

    # primo giro: stabilisce se e quali tools chiamare
    # se non chiama tools, fnisce
    response = client.chat.completions.create(
        model="gpt-4o",    
        messages=messages,
        tools=tools,
        tool_choice="auto"  # fai scegliere al modello se chiamare qualche funzione. puoi forzarlo
    )

    repeated = False # per stampare la lista delle chiamate solo ala prima iterazione

    # loop per continuare a fare chiamate finche finiscno i tools da chiamare
    while True:
        # che tools devo chiamare?
        tool_calls = response.choices[0].message.tool_calls

        if not tool_calls: # se non sono usciti tools o se ho esaurito i tools da azionare....
            
            final_answer = response.choices[0].message.content
            print("Risposta finale dal modello:")
            print(final_answer)
            break
        
        else: # altrimenti... (significa che ci sono (o ci sono ancora) dei tools da azionare)

            # 1) aggiungo il messaggio di function_call ai messaggi (come contesto)
            messages.append(response.choices[0].message)

            if not repeated: # la prima volta, stampo anche lista dei tools da chiamare
                functions_names=[]
                for func in tool_calls:
                    tool_name = func.function.name
                    functions_names.append(str(tool_name))
                print(f"Dovrò chiamare i seguenti tools:\n{functions_names}\n")
                repeated=True # non faccio piu questa parte dopo il primo giro
                
            # 2) eseguo ogni tool_call
            for tc in tool_calls: # per tutti i tools che devo azionare
                
                # ogni tool ha i suoi argomenti, e il modello me li fornisce insieme al nome del tool secondo le spec che gli ho passato
                tool_name = tc.function.name 
                tool_args = json.loads(tc.function.arguments) # serializzo gli argomenti per renderli leggibili dalle funzioni

                if tool_name == "book_flight":
                    result_obj = book_flight(**tool_args)
                elif tool_name == "book_train":
                    result_obj = book_train(**tool_args)
                elif tool_name == "book_hotel":
                    result_obj = book_hotel(**tool_args)
                else:
                    result_obj = {"error": f"Funzione non riconosciuta: {tool_name}"}

                
                print(f"\n") # salto una riga nei print per chiarezza grafica
                
                # converto la risposta del tool di questa iterazione
                # in stringa (il modello si aspetta una stringa)
                result_str = json.dumps(result_obj, ensure_ascii=False)

                # aggiungo il risultato come messaggio di ruolo "tool"
                messages.append({
                    "role": "tool",
                    "tool_call_id": tc.id,  # id della tool call
                    "content": result_str
                })
            # importante: qua finisce il ciclo delle tool call

            # 3) quando ho finito di chiamare i miei tools, passo i risultati ad una nuova chiamata al modello
            #    aggiungendo gli outputs ai messages 
            response = client.chat.completions.create(
                model="gpt-4o",
                messages=messages,
                tools=tools
            )


In [48]:
bot_viaggi("Devo prenotare albergo e treno per una trasferta da bologna a milano con andata il 3 maggio e ritorno il 5 maggio.")

Dovrò chiamare i seguenti tools:
['book_train', 'book_train', 'book_hotel']

Simulazione: effettuo una POST a https://api.simulatedbooking.com/trains con payload:
{'from': 'Bologna', 'to': 'Milano', 'date': '2024-05-03', 'time': '09:00'}


Simulazione: effettuo una POST a https://api.simulatedbooking.com/trains con payload:
{'from': 'Milano', 'to': 'Bologna', 'date': '2024-05-05', 'time': '18:00'}


Simulazione: effettuo una POST a https://api.simulatedbooking.com/hotels con payload:
{'city': 'Milano', 'check_in': '2024-05-03', 'check_out': '2024-05-05', 'rooms': 1}


Risposta finale dal modello:
La trasferta è stata organizzata con successo! Ecco i dettagli:

- **Treno Andata**: Da Bologna a Milano il 3 maggio 2024 alle ore 09:00.
- **Treno Ritorno**: Da Milano a Bologna il 5 maggio 2024 alle ore 18:00.
- **Hotel a Milano**: Prenotato per il soggiorno dal 3 al 5 maggio 2024. Numero di stanze: 1.

Se hai bisogno di ulteriori assistenze o modifiche, fammelo sapere!


In [50]:
bot_viaggi("sorprendimi con un viaggio a sorpresa. decidi tutto te")

Dovrò chiamare i seguenti tools:
['book_flight', 'book_hotel', 'book_train']

Simulazione: effettuo una POST a https://api.simulatedbooking.com/flights con payload:
{'destination': 'Parigi', 'date': '2023-12-15', 'passengers': 1, 'class': 'Economy'}


Simulazione: effettuo una POST a https://api.simulatedbooking.com/hotels con payload:
{'city': 'Parigi', 'check_in': '2023-12-15', 'check_out': '2023-12-18', 'rooms': 1}


Simulazione: effettuo una POST a https://api.simulatedbooking.com/trains con payload:
{'from': 'Parigi Gare de Lyon', 'to': 'Lione Part-Dieu', 'date': '2023-12-16', 'time': '09:00'}


Risposta finale dal modello:
Il tuo viaggio a sorpresa è pronto! Ecco il tuo itinerario:

1. **Volo per Parigi**: Partirai il 15 dicembre 2023 in classe Economy.

2. **Soggiorno in Hotel a Parigi**: Check-in il 15 dicembre e check-out il 18 dicembre. Sarai alloggiato in una stanza singola.

3. **Viaggio in Treno a Lione**: Il 16 dicembre, prenderai un treno da Parigi Gare de Lyon a Lione P